In [ ]:
import pandas as pd
import re

# 1. Load the fixed datasets
fema = pd.read_csv('data/fema_assistance_yearly_summary.csv')
census = pd.read_csv('data/acs_county_social_economic.csv')

# 2. State Mapping (Standardizing Names to Abbreviations)
state_to_abb = {
    'ALABAMA': 'AL', 'ALASKA': 'AK', 'ARIZONA': 'AZ', 'ARKANSAS': 'AR', 'CALIFORNIA': 'CA',
    'COLORADO': 'CO', 'CONNECTICUT': 'CT', 'DELAWARE': 'DE', 'FLORIDA': 'FL', 'GEORGIA': 'GA',
    'HAWAII': 'HI', 'IDAHO': 'ID', 'ILLINOIS': 'IL', 'INDIANA': 'IN', 'IOWA': 'IA',
    'KANSAS': 'KS', 'KENTUCKY': 'KY', 'LOUISIANA': 'LA', 'MAINE': 'ME', 'MARYLAND': 'MD',
    'MASSACHUSETTS': 'MA', 'MICHIGAN': 'MI', 'MINNESOTA': 'MN', 'MISSISSIPPI': 'MS', 'MISSOURI': 'MO',
    'MONTANA': 'MT', 'NEBRASKA': 'NE', 'NEVADA': 'NV', 'NEW HAMPSHIRE': 'NH', 'NEW JERSEY': 'NJ',
    'NEW MEXICO': 'NM', 'NEW YORK': 'NY', 'NORTH CAROLINA': 'NC', 'NORTH DAKOTA': 'ND', 'OHIO': 'OH',
    'OKLAHOMA': 'OK', 'OREGON': 'OR', 'PENNSYLVANIA': 'PA', 'RHODE ISLAND': 'RI', 'SOUTH CAROLINA': 'SC',
    'SOUTH DAKOTA': 'SD', 'TENNESSEE': 'TN', 'TEXAS': 'TX', 'UTAH': 'UT', 'VERMONT': 'VT',
    'VIRGINIA': 'VA', 'WASHINGTON': 'WA', 'WEST VIRGINIA': 'WV', 'WISCONSIN': 'WI', 'WYOMING': 'WY'
}

# 3. The "Heavy Clean" Function for Geography
def geoclean(name):
    if pd.isna(name): return ""
    name = str(name).upper().strip()
    name = re.sub(r'\(.*\)', '', name) # Remove (BOROUGH), (CITY), etc.
    for suffix in [' COUNTY', ' PARISH', ' BOROUGH', ' MUNICIPALITY', ' CITY']:
        name = name.replace(suffix, '')
    return re.sub(r'[^A-Z ]', '', name).strip() # Keep only letters and spaces

# 4. Standardize Join Keys in Both Files
census['state_clean'] = census['state'].str.upper().map(state_to_abb)
census['county_clean'] = census['county'].apply(geoclean)

fema['state_clean'] = fema['State'].str.upper().str.strip()
fema['county_clean'] = fema['county'].apply(geoclean)

# 5. The Master Merge (Inner Join)
# We match on Year, State, and County to ensure the right Census year
# is paired with the right Disaster year.
master_df = pd.merge(
    census,
    fema,
    on=['Year', 'state_clean', 'county_clean'],
    how='inner'
)

# 6. Final Cleanup
# Drop redundant columns if they exist
cols_to_drop = ['state', 'county_y', 'State']
master_df = master_df.drop(columns=[c for c in cols_to_drop if c in master_df.columns])

# Save the final masterpiece
master_df.to_csv('data/master_disaster_data.csv', index=False)

print(f"BOOM! Master Dataset Created.")
print(f"Total Rows: {len(master_df)}")
print(f"Columns: {list(master_df.columns)}")
print(f"Years Covered: {sorted(master_df['Year'].unique())}")

BOOM! Master Dataset Created.
Total Rows: 3611
Columns: ['full_name', 'total_population', 'median_income', 'poverty_count', 'county_x', 'Year', 'poverty_rate', 'state_clean', 'county_clean', 'total_applicants', 'housing_assistance_amount', 'total_assistance_amount']
Years Covered: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
